# Evaluate a Fabric Data Agent with your own ground truth

Companion notebook for the talk *Designing and Operationalizing Data Agents in Microsoft Fabric*.

A data agent answers business questions in natural language. Before you trust it, **test it**: ask it questions whose correct answer you already know, and check what it says. This notebook does exactly that, in four steps:

1. **Compute the ground truth** with plain Spark SQL over the Lakehouse tables. The expected answers come from the data itself, so the test stays valid when the data changes.
2. **Assemble** a table of `question` + `expected_answer`.
3. **Run the evaluation** with the Fabric Data Agent SDK, using a custom *critic prompt* so the check is about correctness, not formatting.
4. **Read the results**: a summary, and the step-by-step detail for every question.

### Prerequisites

- A Fabric workspace with a **published Data Agent** that is connected to the Olist tables (the agent name goes in step 3).
- The public [Brazilian E-Commerce dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) loaded as Delta tables in a Lakehouse: `customers`, `orders`, `order_items`, `order_payments`, `order_reviews`, `products`, `product_category_name_translation`, `sellers`. Check the dataset page for its license. The data is not included in this repository.
- This notebook opened in the same workspace, with that Lakehouse set as its **default Lakehouse**.
- Documentation: [Evaluate your data agent](https://learn.microsoft.com/en-us/fabric/data-science/evaluate-data-agent).

> Outputs were removed on purpose, so the notebook carries no workspace names, ids or run history. Run it yourself to see the results.

In [ ]:
# Install the Fabric Data Agent SDK (it provides the evaluation helpers).
# Run this once per session, then restart the kernel if the notebook asks you to.
%pip install -U fabric-data-agent-sdk

## Step 1. Compute the ground truth from the data

Each query answers one business question and stores a single value (or row) in `results`. Keeping the logic here, rather than typing the expected answers by hand, means a wrong expected answer is a bug you can read and fix.

In [ ]:
import pandas as pd

# Keys of the eight checks below, in the order we will ask them.
QUESTIONS = [
    "avg_review_score",
    "best_category_by_review",
    "late_delivery_count",
    "revenue_2018",
    "state_with_most_sellers",
    "most_common_payment_type",
    "avg_installments_credit_card",
    "unique_customers",
]

results = {}

# 1. Average review score across all orders (a simple aggregation).
results["avg_review_score"] = spark.sql("""
    SELECT ROUND(AVG(review_score), 2) AS v FROM order_reviews
""").first()["v"]

In [ ]:
# 2. Product category with the highest average review score.
#    We require at least 30 reviews so a category with one lucky 5-star
#    review cannot win. This question needs a multi-table join, which is a
#    good test of whether the agent understands the relationships.
results["best_category_by_review"] = spark.sql("""
    SELECT t.product_category_name_english AS category,
           ROUND(AVG(r.review_score), 2) AS avg_score,
           COUNT(*) AS n
    FROM order_reviews r
    JOIN order_items oi ON oi.order_id = r.order_id
    JOIN products p ON p.product_id = oi.product_id
    JOIN product_category_name_translation t
         ON t.product_category_name = p.product_category_name
    GROUP BY t.product_category_name_english
    HAVING COUNT(*) >= 30
    ORDER BY avg_score DESC
    LIMIT 1
""").first()

In [ ]:
# 3. Orders delivered after their estimated delivery date.
#    Orders without a delivery date are excluded (they were not delivered).
results["late_delivery_count"] = spark.sql("""
    SELECT COUNT(*) AS v
    FROM orders
    WHERE order_delivered_customer_date IS NOT NULL
      AND order_delivered_customer_date > order_estimated_delivery_date
""").first()["v"]

In [ ]:
# 4. Total revenue (item price + freight) for orders purchased in 2018.
#    A date filter plus a sum: tests that the agent applies the right year
#    and includes freight, as the question says.
results["revenue_2018"] = spark.sql("""
    SELECT ROUND(SUM(oi.price + oi.freight_value), 2) AS v
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE YEAR(o.order_purchase_timestamp) = 2018
""").first()["v"]

In [ ]:
# 5. State with the most distinct sellers.
#    COUNT(DISTINCT ...) matters: a seller appears in many rows.
results["state_with_most_sellers"] = spark.sql("""
    SELECT seller_state AS state, COUNT(DISTINCT seller_id) AS n
    FROM sellers
    GROUP BY seller_state
    ORDER BY n DESC
    LIMIT 1
""").first()

In [ ]:
# 6. Most common payment type.
results["most_common_payment_type"] = spark.sql("""
    SELECT payment_type, COUNT(*) AS n
    FROM order_payments
    GROUP BY payment_type
    ORDER BY n DESC
    LIMIT 1
""").first()

In [ ]:
# 7. Average installments, credit card payments only (aggregation + filter).
results["avg_installments_credit_card"] = spark.sql("""
    SELECT ROUND(AVG(payment_installments), 2) AS v
    FROM order_payments
    WHERE payment_type = 'credit_card'
""").first()["v"]

In [ ]:
# 8. Unique customers.
#    The dataset has a per-order customer_id and a per-person
#    customer_unique_id. The question is about people, so we count the latter.
#    This is a classic place where an agent can pick the wrong column.
results["unique_customers"] = spark.sql("""
    SELECT COUNT(DISTINCT customer_unique_id) AS v FROM customers
""").first()["v"]

## Step 2. Assemble the question / expected answer set

The SDK expects a DataFrame with two columns: `question` (what a user would ask) and `expected_answer` (what a correct answer must contain). The expected answers include the number *and* the entity, so a wrong category or state is caught too.

In [ ]:
# --- Assemble the question / expected_answer ground-truth set --------------
eval_rows = [
    ("What is the average review score across all orders?",
     f"{results['avg_review_score']}"),

    ("Which product category has the highest average review score?",
     f"{results['best_category_by_review']['category']} "
     f"(avg {results['best_category_by_review']['avg_score']}, "
     f"{results['best_category_by_review']['n']} reviews)"),

    ("How many orders were delivered after their estimated delivery date?",
     f"{results['late_delivery_count']}"),

    ("What was the total revenue, including freight, for orders placed in 2018?",
     f"{results['revenue_2018']}"),

    ("Which state has the most sellers?",
     f"{results['state_with_most_sellers']['state']} "
     f"({results['state_with_most_sellers']['n']} sellers)"),

    ("What is the most common payment type?",
     f"{results['most_common_payment_type']['payment_type']} "
     f"({results['most_common_payment_type']['n']} payments)"),

    ("What is the average number of installments for credit card payments?",
     f"{results['avg_installments_credit_card']}"),

    ("How many unique customers are there?",
     f"{results['unique_customers']}"),
]

## Step 3. Run the evaluation

Set the parameters below, then run. The SDK sends every question to your agent, records the answer and the agent's reasoning steps, and asks a judge model (the *critic*) whether the answer is correct.

In [ ]:
df = pd.DataFrame(eval_rows, columns=["question", "expected_answer"])
display(df)

# --- run the evaluation ---------------------------------------------
from fabric.dataagent.evaluation import evaluate_data_agent

# Name of your published Data Agent (exactly as it appears in the workspace).
data_agent_name = "<your-data-agent-name>"

# (Optional) set this if the Data Agent lives in a different workspace.
workspace_name = None

# Two tables get created: "<table_name>" (summary) and "<table_name>_steps"
# (step-by-step reasoning and trace).
table_name = "olist_evaluation_output"

# "production" (default, the published version) or "sandbox" (unpublished,
# for testing changes before you publish them).
data_agent_stage = "production"

### A custom critic prompt

Numeric and business answers rarely match character for character: the agent may say "R$ 1,234,567.89 in total revenue" while our expected answer is the plain `1234567.89`. A custom critic prompt keeps the check about **correctness, not formatting**. Adjust the tolerance rules to your own domain.

In [ ]:
# Numeric/business answers rarely match character-for-character (the agent
# will say "R$ 1,234,567.89 in total revenue" vs. our plain "1234567.89"), so
# a custom critic_prompt keeps the check about correctness, not formatting.

critic_prompt = """
    Given the following query, expected answer, and actual answer, decide if
    the actual answer is CORRECT relative to the expected answer.

    Treat the answer as correct if:
    - Numbers match within reasonable rounding (e.g. currency formatting,
      1-2 decimal places, thousands separators).
    - The same entity/category/state is identified, even if phrased differently.

    Query: {query}

    Expected Answer:
    {expected_answer}

    Is the actual answer correct? Respond with 'yes' or 'no' and one short
    sentence explaining why.
"""

evaluation_id = evaluate_data_agent(
    df,
    data_agent_name,
    workspace_name=workspace_name,
    table_name=table_name,
    data_agent_stage=data_agent_stage,
    critic_prompt=critic_prompt,
)

In [ ]:
# Keep this id: it identifies this run in the output tables.
print(f"Evaluation run ID: {evaluation_id}")

## Step 4. Read the results

Start with the summary (how many answers were judged correct), then open the details for the questions that failed or were unclear. The detail table includes the agent's reasoning steps, which usually show *why* it went wrong (wrong table, wrong column, a missing instruction).

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_summary

# One row per run: how many answers were true, false or unclear.
summary_df = get_evaluation_summary(table_name, verbose=True)
display(summary_df)

In [ ]:
from fabric.dataagent.evaluation import get_evaluation_details

details_df = get_evaluation_details(
    evaluation_id,
    table_name,
    get_all_rows=True,  # get_all_rows=False -> only rows that came back false/unclear, the ones
                        # worth reading first when you are debugging the agent
    verbose=True,
)
display(details_df)

## What to do with a failure

- **Wrong column or table:** add a clearer table/column description, or an agent instruction ("customers means `customer_unique_id`, not `customer_id`").
- **Wrong logic:** add a worked example query for that question pattern to the agent's example queries.
- **Right answer, wrong format:** loosen the critic prompt, not the agent.
- Re-run with `data_agent_stage = "sandbox"` after each change, and publish only when the run is clean.